In [ ]:
# Safety Fundamentals of Generative AI
# Sharvari 22CS30049
# Part 1

In [1]:
!pip install -q transformers==4.44.0 peft==0.12.0 datasets accelerate bitsandbytes
!pip install -q evaluate rouge_score nltk sacrebleu
!pip install -q mergekit --extra-index-url https://pypi.org/simple/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.

In [ ]:
import os, shutil, json, yaml, subprocess
import torch
import numpy as np
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import evaluate as hf_evaluate


In [ ]:
BASE_MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME     = "medalpaca/medical_meadow_medqa"
MAX_SEQ_LEN      = 512

OUTPUT_DIR_LORA   = "./model_sft_lora"
OUTPUT_DIR_MERGED = "./model_sft_merged"
OUTPUT_DIR_DARE   = "./model_sft_dare"

LORA_R           = 16
LORA_ALPHA       = 32
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = ["q_proj", "k_proj", "v_proj", "o_proj"]


# LEARNING_RATE    = 5e-5
# WARMUP_RATIO     = 0.03
# MAX_SEQ_LEN      = 256
# BATCH_SIZE       = 64
# GRAD_ACCUM       = 8
# NUM_EPOCHS       = 3
# MAP_NUM_PROC     = 1 if os.name == "nt" else 4

LEARNING_RATE    = 2e-4
WARMUP_RATIO     = 0.03
MAX_SEQ_LEN      = 256
BATCH_SIZE       = 8
GRAD_ACCUM       = 2
NUM_EPOCHS       = 2
MAP_NUM_PROC     = 1 if os.name == "nt" else 4

In [10]:
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_NAME,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [11]:
ds_raw = load_dataset(DATASET_NAME)
print(ds_raw["train"].column_names)
print(ds_raw["train"][0])

['input', 'instruction', 'output']
{'input': "Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? \n{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'E: Nitrofurantoin'}


In [12]:
# Use a positional 60/20/20 split.
all_ds = ds_raw['train']
n = len(all_ds)
tr_cut = int(0.6 * n)
va_cut   = int(0.8 * n)

tr_raw = all_ds.select(range(0, tr_cut))
va_raw   = all_ds.select(range(tr_cut, va_cut))
te_raw  = all_ds.select(range(va_cut, n))

print(f"Total: {n} | Train: {len(tr_raw)} | Val: {len(va_raw)} | Test: {len(te_raw)}")

with open("test_indices.json", "w") as f:
    json.dump({"val_end": va_cut, "N": n}, f)

Total: 10178 | Train: 6106 | Val: 2036 | Test: 2036


In [ ]:
# Quick sanity check that the split was not shuffled.
print(all_ds[tr_cut-1]['instruction'][:60])
print(all_ds[tr_cut]['instruction'][:60])

Please answer with one of the option in the bracket
Please answer with one of the option in the bracket


: 

In [13]:
def mk_prompt(instruction: str, input_text: str, output: str = "") -> str:
    if input_text and input_text.strip():
        prompt = (
            f"### Instruction:\n{instruction.strip()}\n\n"
            f"### Input:\n{input_text.strip()}\n\n"
            f"### Response:\n{output.strip()}"
        )
    else:
        prompt = (
            f"### Instruction:\n{instruction.strip()}\n\n"
            f"### Response:\n{output.strip()}"
        )
    return prompt

def fmt_ex(example):
    instruction = example.get('instruction', '') or ''
    input_text  = example.get('input', '') or ''
    output      = example.get('output', '') or ''
    return {"text": mk_prompt(instruction, input_text, output)}

In [15]:
def tok_fn(examples):
    toks = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding="max_length",
    )
    labs = []
    for ids, mask in zip(toks["input_ids"], toks["attention_mask"]):
        lab = [id_ if m == 1 else -100 for id_, m in zip(ids, mask)]
        labs.append(lab)
    toks["labels"] = labs
    return toks

cols_to_remove = tr_raw.column_names

train_ds = (
    tr_raw
    .map(fmt_ex)
    .map(tok_fn, batched=True, remove_columns=cols_to_remove + ["text"])
)
val_ds = (
    va_raw
    .map(fmt_ex)
    .map(tok_fn, batched=True, remove_columns=cols_to_remove + ["text"])
)

train_ds.set_format("torch")
val_ds.set_format("torch")

Map: 100%|██████████| 2036/2036 [00:01<00:00, 1263.13 examples/s]


In [16]:
base_m = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
base_m.config.use_cache = False
base_m.enable_input_require_grads()

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias="none",
)

model = get_peft_model(base_m, lora_cfg)
model.print_trainable_parameters()

`torch_dtype` is deprecated! Use `dtype` instead!
c:\Users\wanja\anaconda3\envs\sdm\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\wanja\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 338/338 [00:06<00:00, 49

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [17]:
train_args = TrainingArguments(
    output_dir=OUTPUT_DIR_LORA,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    dataloader_num_workers=2,
    report_to="none",
    optim="adamw_torch",
    dataloader_pin_memory=False,
    fp16=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
coach = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

fit_out = coach.train()

print("\n=== Training Summary ===")
print(f"  Training loss:  {fit_out.training_loss:.4f}")
print(f"  Total steps:    {fit_out.global_step}")
print(f"  Runtime:        {fit_out.metrics['train_runtime']:.1f}s")

Starting training...


Epoch,Training Loss,Validation Loss
1,1.290969,1.263778
2,1.231393,1.251453



=== Training Summary ===
  Training loss:  1.2868
  Total steps:    764
  Runtime:        14790.1s


In [12]:
coach.save_model(OUTPUT_DIR_LORA)
tokenizer.save_pretrained(OUTPUT_DIR_LORA)

with open(os.path.join(OUTPUT_DIR_LORA, "training_metrics.json"), "w") as f:
    json.dump(fit_out.metrics, f, indent=2)

model_sft_lora saved at: ./model_sft_lora
Contents: ['chat_template.jinja', 'README.md', 'adapter_config.json', 'training_args.bin', 'checkpoint-764', 'adapter_model.safetensors', 'checkpoint-382', 'tokenizer_config.json', 'tokenizer.json', 'training_metrics.json']


In [ ]:
# Merge LoRA into the full model before DARE.
# mergekit needs full weights, not adapter-only checkpoints.
# Then try drop rates 0.1, 0.3, 0.5, and 0.7.

In [14]:
# Merge LoRA into the base model.
base_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

peft_m = PeftModel.from_pretrained(base_merge, OUTPUT_DIR_LORA)
merged_m = peft_m.merge_and_unload()

os.makedirs(OUTPUT_DIR_MERGED, exist_ok=True)
merged_m.save_pretrained(OUTPUT_DIR_MERGED, safe_serialization=True)
tokenizer.save_pretrained(OUTPUT_DIR_MERGED)

del peft_m, base_merge
torch.cuda.empty_cache()

Loading base model for LoRA merge...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying LoRA adapters...
Merging LoRA weights into base...
Saving merged model to ./model_sft_merged...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved. Contents: ['model.safetensors', 'chat_template.jinja', 'config.json', 'tokenizer_config.json', 'generation_config.json', 'tokenizer.json']


In [15]:

DROP_RATES = [0.1, 0.3, 0.5, 0.7]

def dare_cfg(sft_model_path: str, base_model_name: str, dens: float) -> dict:
    """Build the mergekit config for DARE-linear."""
    return {
        "merge_method": "dare_linear",
        "base_model": base_model_name,
        "models": [
            {
                "model": sft_model_path,
                "parameters": {
                    "weight": 1.0,
                    "density": dens,
                }
            }
        ],
        "parameters": {
            "normalize": False,
            "int8_mask": False,
        },
        "dtype": "bfloat16"
    }

dare_dirs = {}

for p in DROP_RATES:
    dens = round(1.0 - p, 2)
    out_dir  = f"./dare_candidate_p{str(p).replace('.', '')}"
    cfg_file = f"./dare_config_p{str(p).replace('.', '')}.yaml"

    cfg = dare_cfg(OUTPUT_DIR_MERGED, BASE_MODEL_NAME, dens)
    with open(cfg_file, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

    res = subprocess.run(
        ["mergekit-yaml", cfg_file, out_dir,
         "--trust-remote-code", "--allow-crimes"],
        capture_output=True, text=True
    )

    if res.returncode == 0:
        dare_dirs[p] = out_dir
    else:
        raise RuntimeError(res.stderr[-500:])

print("DARE candidates generated:", dare_dirs)


>>> Running DARE with drop_rate p=0.1 (density=0.9)...
    ✓ Saved to ./dare_candidate_p01

>>> Running DARE with drop_rate p=0.3 (density=0.7)...
    ✓ Saved to ./dare_candidate_p03

>>> Running DARE with drop_rate p=0.5 (density=0.5)...
    ✓ Saved to ./dare_candidate_p05

>>> Running DARE with drop_rate p=0.7 (density=0.3)...
    ✓ Saved to ./dare_candidate_p07

All DARE candidates generated: {0.1: './dare_candidate_p01', 0.3: './dare_candidate_p03', 0.5: './dare_candidate_p05', 0.7: './dare_candidate_p07'}


In [16]:
rouge_metric  = hf_evaluate.load("rouge")
meteor_metric = hf_evaluate.load("meteor")
bleu_metric   = hf_evaluate.load("bleu")

def infer_prompt(example: dict) -> str:
    """Build a prompt without the answer."""
    instruction = example.get('instruction', '') or ''
    input_text  = example.get('input', '') or ''
    return mk_prompt(instruction, input_text, output="")

def eval_val(
    model_path: str,
    val_examples,
    num_samples: int = 200,
    max_new_tokens: int = 128,
    batch_size: int = 8
) -> dict:
    """Generate responses and compute ROUGE-L, METEOR, and BLEU."""
    eval_m = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16,
        device_map="auto", trust_remote_code=True
    )
    eval_tokz = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if eval_tokz.pad_token is None:
        eval_tokz.pad_token = eval_tokz.eos_token
    eval_tokz.padding_side = "left"
    eval_m.eval()

    sample_ds = val_examples.select(range(min(num_samples, len(val_examples))))
    prompts    = [infer_prompt(ex) for ex in sample_ds]
    refs = [ex['output'] or '' for ex in sample_ds]

    preds = []
    for i in range(0, len(prompts), batch_size):
        batch_ps = prompts[i:i+batch_size]
        inputs = eval_tokz(
            batch_ps, return_tensors="pt",
            padding=True, truncation=True,
            max_length=MAX_SEQ_LEN - max_new_tokens
        ).to(eval_m.device)

        with torch.no_grad():
            outs = eval_m.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=eval_tokz.pad_token_id,
                eos_token_id=eval_tokz.eos_token_id,
            )

        for out in outs:
            input_len = inputs["input_ids"].shape[1]
            gen = eval_tokz.decode(out[input_len:], skip_special_tokens=True)
            preds.append(gen.strip())

    rouge_sc = rouge_metric.compute(preds=preds, refs=refs)
    meteor_sc = meteor_metric.compute(preds=preds, refs=refs)
    bleu_sc   = bleu_metric.compute(
        preds=preds,
        refs=[[r] for r in refs]
    )

    del eval_m
    torch.cuda.empty_cache()

    return {
        "rougeL": round(rouge_sc['rougeL'], 4),
        "meteor": round(meteor_sc['meteor'], 4),
        "bleu": round(bleu_sc['bleu'], 4),
    }

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


Evaluation function ready.


In [17]:
dare_res = {}

for p, model_dir in dare_dirs.items():
    print(f"\n=== Evaluating DARE p={p} ===")
    sc = eval_val(model_dir, va_raw, num_samples=200)
    dare_res[p] = sc
    print(f"  ROUGE-L: {sc['rougeL']:.4f} | METEOR: {sc['meteor']:.4f} | BLEU: {sc['bleu']:.4f}")

print("\n=== DARE Sweep Results ===")
print(f"{'Drop Rate':>12} | {'ROUGE-L':>8} | {'METEOR':>8} | {'BLEU':>8}")
print("-" * 48)
for p, s in sorted(dare_res.items()):
    print(f"  p = {p:<8} | {s['rougeL']:>8.4f} | {s['meteor']:>8.4f} | {s['bleu']:>8.4f}")


=== Evaluating DARE p=0.1 ===
  Loading ./dare_candidate_p01...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    Processed 40/200 samples
    Processed 80/200 samples
    Processed 120/200 samples
    Processed 160/200 samples
    Processed 200/200 samples
  ROUGE-L: 0.0368 | METEOR: 0.0841 | BLEU: 0.0037

=== Evaluating DARE p=0.3 ===
  Loading ./dare_candidate_p03...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    Processed 40/200 samples
    Processed 80/200 samples
    Processed 120/200 samples
    Processed 160/200 samples
    Processed 200/200 samples
  ROUGE-L: 0.0388 | METEOR: 0.0915 | BLEU: 0.0038

=== Evaluating DARE p=0.5 ===
  Loading ./dare_candidate_p05...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    Processed 40/200 samples
    Processed 80/200 samples
    Processed 120/200 samples
    Processed 160/200 samples
    Processed 200/200 samples
  ROUGE-L: 0.0369 | METEOR: 0.0919 | BLEU: 0.0038

=== Evaluating DARE p=0.7 ===
  Loading ./dare_candidate_p07...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

    Processed 40/200 samples
    Processed 80/200 samples
    Processed 120/200 samples
    Processed 160/200 samples
    Processed 200/200 samples
  ROUGE-L: 0.0371 | METEOR: 0.0871 | BLEU: 0.0037

=== DARE Sweep Results ===
   Drop Rate |  ROUGE-L |   METEOR |     BLEU
------------------------------------------------
  p = 0.1      |   0.0368 |   0.0841 |   0.0037
  p = 0.3      |   0.0388 |   0.0915 |   0.0038
  p = 0.5      |   0.0369 |   0.0919 |   0.0038
  p = 0.7      |   0.0371 |   0.0871 |   0.0037


In [18]:
best_drop = max(dare_res, key=lambda p: dare_res[p]['rougeL'])
best_sc = dare_res[best_drop]

print(f"Best drop rate: p = {best_drop}")
print(f"  ROUGE-L: {best_sc['rougeL']:.4f}")
print(f"  METEOR:  {best_sc['meteor']:.4f}")
print(f"  BLEU:    {best_sc['bleu']:.4f}")

best_dir = dare_dirs[best_drop]
if os.path.exists(OUTPUT_DIR_DARE):
    shutil.rmtree(OUTPUT_DIR_DARE)
shutil.copytree(best_dir, OUTPUT_DIR_DARE)

pick_meta = {
    "best_drop_rate": best_drop,
    "selection_metric": "ROUGE-L",
    "all_results": {str(p): s for p, s in dare_res.items()},
    "best_scores": best_sc,
}
with open("dare_selection.json", "w") as f:
    json.dump(pick_meta, f, indent=2)

Best drop rate: p = 0.3
  ROUGE-L: 0.0388
  METEOR:  0.0915
  BLEU:    0.0038

✓ model_sft_dare saved at: ./model_sft_dare
  Contents: ['model.safetensors', 'README.md', 'config.json', 'mergekit_config.yml', 'merges.txt', 'tokenizer_config.json', 'tokenizer.json']
